# stVCR synthetic data training

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Fit stVCR and export forecasts

In [ ]:
# Synthetic Developmental Crescent Tissue v1 (seed 20261023)
# 1) Official stVCR training defaults, with upstream AE + initial and dynamic rigid alignment.
# 2) Pre-registered sensitivity arm: kappa_exp=alpha_exp=0.005 vs defaults=0.02.
# Both arms use IDENTICAL frozen public observations and one common AE; neither reads oracle truth.
# Source-input probe set: the generator's immutable 363 source-tissue-2 IDs.
# Time t*=2.5/9 held out from all fitting; outputs are predictions, not test scores.
# For a data/installation preflight without training: set STVCR_PREFLIGHT_ONLY=1.

import copy, gc, hashlib, importlib, json, os, random, subprocess, sys, time, types
from pathlib import Path

import numpy as np
import pandas as pd

# ========== ONLY EDIT THESE PATHS (if your Skipjack checkout differs) ==========
PROJECT = Path(os.environ.get('STP_PROJECT',
    os.environ["STP_PROJECT"]))
EXP = PROJECT / 'Experiments/simulation validation/Synthetic benchmarking/experiments'
DATA_ROOT = Path(os.environ.get('CRESCENT_DATA_ROOT',
    str(EXP / 'synthetic_developmental_crescent_v1')))
STVCR_REPO = Path(os.environ.get('STVCR_REPO',
    os.environ["STVCR_REPO"]))
OUT_ROOT = Path(os.environ.get('STVCR_CRESCENT_OUT',
    str(PROJECT / 'experiments/synthetic_benchmark/synthetic_developmental_crescent_v1_stvcr')))
RUNS = ('official_defaults', 'low_expression_weights')  # Both; or run one first.
PRE_FLIGHT = os.environ.get('STVCR_PREFLIGHT_ONLY','0') == '1'
EXPECTED_REPO_COMMIT = os.environ.get('STVCR_EXPECTED_COMMIT',
    '26aa79a63eba7a5e21726b1eb95bf6bb61cfe699')
ALLOW_DIFFERENT_UPSTREAM = os.environ.get('STVCR_ALLOW_DIFFERENT_UPSTREAM','0') == '1'
# ============================================================================
SOURCE_TIME = 2/9
TARGET_TIME = 2.5/9
DGP_SEED = 20261023
AE_SEED = 19491001               # upstream stVCR default
TRAIN_SEED = 20261023             # identical between the two dynamical runs
GENES = [f'G{i:02d}' for i in range(1, 49)]
PUBLIC = DATA_ROOT / 'train_only/public'
MANIFEST = DATA_ROOT / 'manifests/dataset_manifest.json'
PROBE_FILE = DATA_ROOT / 'manifests/source_probe_cell_ids.csv.gz'


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for c in iter(lambda: f.read(1<<20), b''):
            h.update(c)
    return h.hexdigest()


def dump_json(path, data):
    path = Path(path)
    tmp = path.with_name(path.name + '.partial')
    tmp.write_text(json.dumps(data, sort_keys=True, indent=2, default=str) + '\n')
    os.replace(tmp, path)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# -------------------------------- PUBLIC DATA --------------------------------
assert MANIFEST.is_file(), f'Missing frozen generator manifest: {MANIFEST}'
manifest = json.loads(MANIFEST.read_text())
assert manifest['dataset_id'] == 'synthetic_developmental_crescent_v1'
assert manifest['seed'] == DGP_SEED and manifest['status'] == 'frozen'
assert manifest['n_observed_cells'] == 36425 and manifest['n_heldout_cells'] == 2597
assert manifest['n_source_probes'] == 363 and manifest['genes'] == GENES
assert np.isclose(manifest['source_t'], SOURCE_TIME)
assert np.isclose(manifest['target_t'], TARGET_TIME)
assert len(manifest['tissue_times']) == 10
# Hash only public inputs and manifest's source-probe file. Never open evaluator_only/.
PUBLIC_RELS = ('train_only/public/observed_cells.csv.gz',
               'train_only/public/counts.csv.gz',
               'train_only/public/genes.csv',
               'manifests/source_probe_cell_ids.csv.gz')
verified = {}
for rel in PUBLIC_RELS:
    p = DATA_ROOT / rel
    assert p.is_file(), f'Missing public input: {p}'
    actual = sha256(p)
    expected = manifest['file_sha256'].get(rel)
    assert expected and actual == expected, f'Dataset checksum differs: {rel}'
    verified[rel] = actual

obs = pd.read_csv(PUBLIC/'observed_cells.csv.gz')
counts = pd.read_csv(PUBLIC/'counts.csv.gz')
probe_ids = pd.read_csv(PROBE_FILE).cell_id.astype(str).to_numpy()
genes_file = pd.read_csv(PUBLIC/'genes.csv').gene.astype(str).tolist()
assert genes_file == GENES
assert list(obs.columns) == ['cell_id','tissue_id','time','x','y']
assert list(counts.columns) == ['cell_id',*GENES]
assert len(obs) == len(counts) == 36425
assert obs.cell_id.is_unique and counts.cell_id.is_unique
assert np.array_equal(obs.cell_id.to_numpy(), counts.cell_id.to_numpy()), 'Cell order mismatch'
assert not np.isclose(obs.time.to_numpy(float), TARGET_TIME).any(), 'TARGET LEAKAGE'
assert np.allclose(np.sort(obs.time.unique()),np.linspace(0,1,10))
assert len(probe_ids) == 363 and len(set(probe_ids)) == 363
index = pd.Index(obs.cell_id.astype(str))
probe_ix = index.get_indexer(probe_ids)
assert (probe_ix>=0).all() and (obs.iloc[probe_ix].tissue_id.to_numpy()==2).all()
assert np.isclose(obs.iloc[probe_ix].time.to_numpy(),SOURCE_TIME).all()
X = counts[GENES].to_numpy(dtype=np.float32, copy=True)
assert np.isfinite(X).all() and np.all(X>=0) and np.array_equal(X,np.round(X))
XY = obs[['x','y']].to_numpy(dtype=np.float32)
assert np.isfinite(XY).all()
assert X.shape == (36425,48)
print('LOCKED DATASET:', DATA_ROOT)
print('Observed input cells:',len(obs),'| genes:',X.shape[1], '| source probes:',len(probe_ids))
print('Fit time grid:',np.sort(obs.time.unique()),'| held-out t*:',TARGET_TIME)
print('Dataset checksums match the generator manifest. No hidden/oracle data opened.')
if PRE_FLIGHT:
    print('PREFLIGHT PASSED. STVCR_PREFLIGHT_ONLY=1; no training or predictions were run.')
else:
    # --------------------------- UPSTREAM IMPORT ----------------------------
    import torch
    import anndata as ad
    import matplotlib.pyplot as plt
    from torch.utils.data import DataLoader
    from torchdiffeq import odeint
    from IPython.display import display
    assert torch.cuda.is_available(), 'Request a Skipjack GPU and use the stVCR kernel.'
    DEVICE = torch.device('cuda:0')
    torch.set_num_threads(min(4,max(1,os.cpu_count() or 1)))
    src = STVCR_REPO / 'src/stvcr'
    assert (src/'training/train.py').is_file() and (src/'preprocessing/pp.py').is_file(), STVCR_REPO
    # Import unmodified upstream files, bypassing only optional package-level imports.
    if 'stvcr' in sys.modules:
        assert str(src) in [str(p) for p in sys.modules['stvcr'].__path__], (
            'A different stVCR installation is imported: restart Jupyter kernel.')
    else:
        p=types.ModuleType('stvcr');p.__path__=[str(src)];p.__package__='stvcr'
        sys.modules['stvcr']=p
    for sub in ('training','preprocessing'):
        key='stvcr.'+sub
        if key not in sys.modules:
            p=types.ModuleType(key);p.__path__=[str(src/sub)];p.__package__=key
            sys.modules[key]=p
    ae_utils = importlib.import_module('stvcr.preprocessing.ae_utils')
    pp = importlib.import_module('stvcr.preprocessing.pp')
    train = importlib.import_module('stvcr.training.train')
    git_sha = subprocess.check_output(['git','-C',str(STVCR_REPO),'rev-parse','HEAD'],
                                      stderr=subprocess.DEVNULL,text=True).strip()
    if git_sha != EXPECTED_REPO_COMMIT and not ALLOW_DIFFERENT_UPSTREAM:
        raise RuntimeError(f'Pinned upstream commit mismatch: {git_sha} != {EXPECTED_REPO_COMMIT}. '
                           'Do not change code silently. To explicitly test another upstream revision, '
                           'set STVCR_ALLOW_DIFFERENT_UPSTREAM=1 and record the SHA.')
    assert int(train.default_config['n_epochs']) == 2000
    assert int(train.default_config['num_samples']) == 500
    assert float(train.default_config['kappa_exp']) == .02
    assert float(train.default_config['alpha_exp']) == .02
    print('GPU:',torch.cuda.get_device_name(0),'| stVCR commit:',git_sha)
    print('Upstream train_base is NOT patched; training via original train_stvcr().')
    # Official-source inputs are the true observed x,y. Vertical z-up flip was PLOT ONLY.
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    COMMON = OUT_ROOT / 'shared_official_preprocessing'
    COMMON.mkdir(parents=True, exist_ok=True)
    AE_FILE = COMMON/'upstream_ae_best.pt'
    PP_FILE = COMMON/'stvcr_preprocessed_public.h5ad'
    PP_DONE = COMMON/'preprocessing_complete.json'
    repo_note = {'dataset_id': manifest['dataset_id'],'seed':DGP_SEED,
                 'dataset_public_sha256': verified,'upstream_commit':git_sha,
                 'torch':torch.__version__,'numpy':np.__version__,
                 'python':sys.version.split()[0],
                 'gene_scale':'raw observed Poisson counts (48 genes)',
                 'preprocessing_note': 'Disable Scanpy filtering/HVG for fixed 48-gene panel; '
                       'use author AE training and pp_init rigid alignment, both with official defaults.',
                 'alignment':'author pp_init initial alignment ON (alpha=0.002, 5000 pairs); '
                       'author train_stvcr dynamic rigid alignment ON; NO oracle pose correction',
                 'AE': {'latent_dim':10,'epochs_max':1000,'lr':.001,'batch':1000,
                        'early_stop':30,'validation_fraction':.1,'seed':AE_SEED},
                 'dynamics': {'epochs':2000,'num_samples_initial':500,
                              'growth':True,'alignment':True},
                 'source_probes':'original observed cells; future values unseen',
                 'held_out':'independent unseen time 2.5/9, not a leave-one-snapshot-out test'}
    if PP_DONE.exists():
        old=json.loads(PP_DONE.read_text())
        assert old['protocol']==repo_note, 'Existing preprocessing manifest differs; move/archive output.'
        assert sha256(AE_FILE)==old['ae_sha256'] and sha256(PP_FILE)==old['preprocessed_sha256']
        print('Reusing verified upstream AE + preprocessing.')
        a=ad.read_h5ad(PP_FILE)
    else:
        if AE_FILE.exists() or PP_FILE.exists():
            raise RuntimeError('Partial preprocessing artifacts exist. Inspect/archive before a clean run: '+str(COMMON))
        # Official AE implementation + official default parameters.
        seed_all(AE_SEED)
        model_ae=ae_utils.AE(n_genes=len(GENES),z_dims=10).to(DEVICE)
        ds=ae_utils.DatasetAE(torch.from_numpy(X))
        tr_ds,val_ds=ae_utils.train_valid_split(ds,valid_ratio=.1,seed=AE_SEED)
        tr_loader=DataLoader(tr_ds,batch_size=1000,shuffle=True)
        val_loader=DataLoader(val_ds,batch_size=1000,shuffle=False)
        print('Training authors\' AE, best-validation checkpoint saved to:',AE_FILE,flush=True)
        ae_utils.train_ae(model_ae,tr_loader,val_loader,
            {'seed':AE_SEED,'learning_rate':.001,'n_epochs':1000,
             'batch_size':1000,'early_stop':30,'valid_ratio':.1,
             'save_path':str(AE_FILE)},DEVICE)
        # IMPORTANT: official ae_dim_reduction() retains final in-memory weights in X_ae
        # but writes BEST validation checkpoint. Encode from the BEST checkpoint for consistency.
        best_ae=torch.load(AE_FILE,map_location=DEVICE,weights_only=False)
        best_ae.eval()
        encoded=np.empty((len(X),10),dtype=np.float32)
        recon_sqerr=0.; n_entries=0
        with torch.inference_mode():
            for j in range(0,len(X),2048):
                xb=torch.from_numpy(X[j:j+2048]).to(DEVICE)
                zz=best_ae.encode_mlp1(xb)
                pred=best_ae.decode_mlp1(zz)
                encoded[j:j+len(xb)]=zz.cpu().numpy()
                recon_sqerr+=float(torch.square(pred-xb).sum().item())
                n_entries+=xb.numel()
        ae_train_rmse=float(np.sqrt(recon_sqerr/n_entries))
        print(f'AE observed-data reconstruction RMSE: {ae_train_rmse:.5f} (raw counts)',flush=True)
        del model_ae,best_ae
        gc.collect();torch.cuda.empty_cache()
        # The 48-gene panel is ALL used; upstream Scanpy min_genes=100 would remove all cells.
        a=ad.AnnData(X=X.copy(),
            obs=pd.DataFrame({'cell_id':obs.cell_id.astype(str).to_numpy(),
                              'time':obs.time.to_numpy(dtype=float),
                              'tissue_id':obs.tissue_id.to_numpy(dtype=int)},
                             index=pd.Index(obs.cell_id.astype(str),name='cell_id')))
        a.var_names=GENES
        a.obsm['spatial']=XY.copy()
        a.obsm['X_ae']=encoded
        # EXACT authors' coordinate preprocessing: center each slice, perform OT-informed
        # rigid initial alignment, normalize common spatial scale and normalize time.
        print('Running upstream pp_init (initial alignment=True, alpha=.002, n=5000).',flush=True)
        pp.pp_init(a, spatial_key='spatial',gene_redunction_key='X_ae',time_key='time',
                   use_initial_alignment=True,alpha=.002,down_sampling_number=5000,
                   normlize_spatial_coordinate=True)
        assert a.obsm['X_gene_input'].shape==(len(X),10)
        assert a.obsm['X_spatial_input'].shape==(len(X),2)
        assert np.isfinite(a.obsm['X_spatial_input']).all()
        assert np.isfinite(a.obsm['X_gene_input']).all()
        assert np.allclose(a.obs.time_input.to_numpy(),a.obs.time.to_numpy())
        # Persist only public observations + author-derived embeddings and alignment.
        a.write_h5ad(PP_FILE)
        dump_json(PP_DONE,{'protocol':repo_note,
                           'ae_sha256':sha256(AE_FILE),
                           'preprocessed_sha256':sha256(PP_FILE),
                           'ae_observed_rmse':ae_train_rmse})
        print('Saved shared preprocessing:',PP_FILE,flush=True)
    # -----------------------------------------------------------------------
    assert a.n_obs==36425 and list(a.var_names)==GENES
    assert np.array_equal(a.obs_names.to_numpy(),obs.cell_id.astype(str).to_numpy())
    assert np.allclose(a.obs.time_input.to_numpy(),obs.time.to_numpy())
    source_mask=a.obs.tissue_id.to_numpy()==2
    all_src_ids=a.obs_names.to_numpy()[source_mask].astype(str)
    all_src_xy=np.asarray(a.obsm['X_spatial_input'][source_mask],dtype=np.float32)
    all_src_latent=np.asarray(a.obsm['X_gene_input'][source_mask],dtype=np.float32)
    probe_select=pd.Index(all_src_ids).get_indexer(probe_ids)
    assert (probe_select>=0).all()
    # QC author alignment: plot tissue-2/3 observed vs author-registered coordinates.
    fig,axes=plt.subplots(1,2,figsize=(10.5,4.6))
    for k in (2,3):
        ids_mask=a.obs.tissue_id.to_numpy()==k
        color=f'C{k-2}'
        axes[0].scatter(XY[ids_mask,0],XY[ids_mask,1],s=.55,alpha=.40,c=color,label=f'tissue {k}')
        coords=np.asarray(a.obsm['X_spatial_input'][ids_mask])
        axes[1].scatter(coords[:,0],coords[:,1],s=.55,alpha=.40,c=color,label=f'tissue {k}')
    axes[0].set_title('Original observed sequencer x,y (no oracle correction)')
    axes[1].set_title('Authors\' pp_init aligned and scaled coordinates')
    for ax in axes:
        ax.set_aspect('equal');ax.invert_yaxis();ax.legend(markerscale=5,frameon=False,fontsize=8)
    fig.tight_layout()
    fig.savefig(COMMON/'initial_alignment_qc.png',dpi=200)
    display(fig);plt.close(fig)

    for run_name in RUNS:
        assert run_name in ('official_defaults','low_expression_weights')
        run_dir=OUT_ROOT/run_name
        run_dir.mkdir(parents=True,exist_ok=True)
        cfg=copy.deepcopy(train.default_config)
        if run_name=='low_expression_weights':
            cfg['kappa_exp']=.005
            cfg['alpha_exp']=.005
        # Do not override any other authors' recommended values.
        protocol={'shared_preprocessing_sha256':sha256(PP_FILE),
                  'upstream_commit':git_sha,'run_name':run_name,
                  'config':cfg,'train_seed':TRAIN_SEED,
                  'growth':True,'dynamic_rigid_alignment':True}
        RUN_MANIFEST=run_dir/'run_config.json'
        COMPLETE=run_dir/'complete.json'
        MODEL=run_dir/'stvcr_model_2000.pt'
        RIGID=run_dir/'stvcr_rigid_2000.pt'
        FORECAST=run_dir/'stvcr_t2_to_t2p5_frozen_forecast.npz'
        if RUN_MANIFEST.exists():
            assert json.loads(RUN_MANIFEST.read_text())==protocol, 'Changed run configuration'
        else:
            dump_json(RUN_MANIFEST,protocol)
        if COMPLETE.exists():
            d=json.loads(COMPLETE.read_text())
            for filename,expected in d['sha256'].items():
                path=run_dir/filename
                assert path.exists() and sha256(path)==expected, 'Invalid completed artifact: '+filename
            print(f'[{run_name}] verified completed run, no retraining.',flush=True)
        else:
            if MODEL.exists() or RIGID.exists() or FORECAST.exists():
                raise RuntimeError(f'[{run_name}] found incomplete/partial files. '
                    'Upstream stVCR cannot resume its optimizer. Archive/move partial run before retry.')
            seed_all(TRAIN_SEED)
            print(f'\n[{run_name}] starting authors\' train_stvcr, n_epochs={cfg["n_epochs"]}, '
                  f'kappa={cfg["kappa_exp"]}, alpha={cfg["alpha_exp"]}; alignment ON, growth ON',flush=True)
            start=time.perf_counter()
            # NOTE: Passing adata copy avoids mutated aligned obs state between arms.
            adata_fit=a.copy()
            train.train_stvcr(adata=adata_fit,model_path=str(MODEL),
                rigid_transformation_path=str(RIGID),
                config=copy.deepcopy(cfg),use_gene=True,use_spatial=True,
                use_growth=True,use_alignment=True,device=DEVICE,
                gene_input_key='X_gene_input',spatial_input_key='X_spatial_input',
                time_input_key='time_input')
            torch.cuda.synchronize()
            duration=time.perf_counter()-start
            assert MODEL.is_file() and RIGID.is_file()
            del adata_fit;gc.collect();torch.cuda.empty_cache()
            print(f'[{run_name}] training finished in {duration/60:.1f} minutes; forecasting...',flush=True)
            # ----- Forecast each observed source cell at the strictly unseen time -----
            # File names are standard evaluator adapters; no hidden truth is opened.
            best_ae=torch.load(AE_FILE,map_location=DEVICE,weights_only=False)
            dyn=torch.load(MODEL,map_location=DEVICE,weights_only=False)
            rigid=torch.load(RIGID,map_location=DEVICE,weights_only=False)
            best_ae.eval();dyn.eval();rigid.eval()
            states=[]
            aligned_source=[]
            with torch.inference_mode():
                for j in range(0,len(all_src_ids),256):
                    s=torch.as_tensor(all_src_xy[j:j+256],dtype=torch.float32,device=DEVICE)
                    z=torch.as_tensor(all_src_latent[j:j+256],dtype=torch.float32,device=DEVICE)
                    # train_stvcr dynamically aligns later snapshots; tissue index=2.
                    s_aligned=rigid(s,2)
                    aligned_source.append(s_aligned.detach().cpu().numpy())
                    initial=torch.cat([s_aligned,z],dim=1)
                    def rhs(t,u):
                        return torch.cat([dyn.spatial_velocity_net(t,u),
                                          dyn.gene_velocity_net(t,u)],dim=-1)
                    y=odeint(rhs,initial,
                             torch.tensor([SOURCE_TIME,TARGET_TIME],dtype=torch.float32,device=DEVICE),
                             rtol=2e-5,atol=1e-6,method='dopri5',options={'max_num_steps':10000})[-1]
                    states.append(y.detach().cpu().numpy())
            endpoint=np.vstack(states).astype(np.float32)
            aligned_source=np.vstack(aligned_source).astype(np.float32)
            assert endpoint.shape==(len(all_src_ids),12) and np.isfinite(endpoint).all()
            raw_forecast=np.empty((len(all_src_ids),48),dtype=np.float32)
            with torch.inference_mode():
                for j in range(0,len(endpoint),1024):
                    latent=torch.from_numpy(endpoint[j:j+1024,2:]).to(DEVICE)
                    raw_forecast[j:j+len(latent)]=best_ae.decode_mlp1(latent).cpu().numpy()
            assert np.isfinite(raw_forecast).all()
            pred=np.maximum(raw_forecast,0.)
            assert pred.shape==(len(all_src_ids),48)
            spatial_scale=float(a.uns['spatial_scale_factor'])
            np.savez_compressed(FORECAST,
                source_ids=all_src_ids, source_probe_ids=probe_ids,
                source_probe_mask=np.isin(all_src_ids,probe_ids),
                genes=np.asarray(GENES),
                t_source=np.asarray(SOURCE_TIME),t_pred=np.asarray(TARGET_TIME),
                pred_counts_proxy_raw=raw_forecast,
                pred_counts_proxy_nonnegative=pred,
                pred_xy_aligned_normalized=endpoint[:,:2],
                pred_xy_aligned_scaled=endpoint[:,:2]*spatial_scale,
                source_xy_initial_aligned_normalized=aligned_source,
                source_xy_pre_dynamics_alignment_normalized=all_src_xy,
                model_variant=np.array(run_name))
            # Done marker only after all required artifacts are saved.
            done={'variant':run_name,'upstream_commit':git_sha,
                  'training_duration_s':round(duration,2),
                  'source_all_n':int(len(all_src_ids)),
                  'source_probes_n':int(np.isin(all_src_ids,probe_ids).sum()),
                  'train_times':sorted(set(map(float,a.obs.time_input))),
                  't_star':TARGET_TIME,
                  'sha256':{path.name:sha256(path) for path in (MODEL,RIGID,FORECAST)}}
            dump_json(COMPLETE,done)
            del dyn,rigid,best_ae;gc.collect();torch.cuda.empty_cache()
            print(f'[{run_name}] forecast saved:',FORECAST,flush=True)
        # No scoring against simulator truth here. Only plot public-input forecasts.
        prediction=np.load(FORECAST,allow_pickle=False)
        selected=prediction['source_probe_mask']
        assert selected.sum()==363
        print(f'[{run_name}] mean predicted expression share (probes):',
              np.round(100 * (prediction['pred_counts_proxy_nonnegative'][selected].mean(0) /
                    prediction['pred_counts_proxy_nonnegative'][selected].mean(0).sum())[:5],3))
        fig,ax=plt.subplots(figsize=(5.7,4.5))
        sx=prediction['source_xy_initial_aligned_normalized'][selected]
        ex=prediction['pred_xy_aligned_normalized'][selected]
        ax.scatter(sx[:,0],sx[:,1],s=6,alpha=.45,label='source probe t=2/9')
        ax.scatter(ex[:,0],ex[:,1],s=6,alpha=.45,label='stVCR forecast t=2.5/9')
        ax.invert_yaxis();ax.set_aspect('equal');ax.legend(loc='best',markerscale=2,frameon=False)
        ax.set_title(run_name.replace('_',' ')+' — input and forecast')
        fig.tight_layout();fig.savefig(run_dir/'source_probe_forecast_qc.png',dpi=200)
        display(fig);plt.close(fig)
    print('\nDONE: both stVCR variants use identical locked observations and author preprocessing.')
    print('Public forecast artifacts are under:',OUT_ROOT)
    print('IMPORTANT: no evaluator_only files have been read; evaluate separately after freezing forecasts.')